# MarginOps - Guest Review Data Cleaning 

# Objective 
Clean and validate reviews that guests have created per site before analysis in SQL and Excel; visualising data in Power BI covering performance KPIs. 

# Business context 
This dataset contains site names, dates, source of the reviews, the star rating and review text. 

# Cleaning principles

- Raw data is never overwritten. 
- Ambiguous values are preserved where possible.
- Duplicate records are investigated before removal. 
- Imputed values are labelled. 
- Exclusions are documented.

# Importing libraries, configurations and loading raw data

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np 
import seaborn as sns 

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "cleaning":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "guest_reviews.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "cleaned" / "guest_reviews_cleaned.csv"

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}") 

In [ ]:
df.info()
display(df.head())
display(df.tail())
display(df.describe(include="all").T)
display(df.isna().sum().rename("missing_values"))
display(df.nunique().rename("unique_values"))

## Initial observations 

- The raw dataset contains 1,171 rows and 5 columns of data.
- Recorded dates range from the 1st of August 2024 to the 17th of August 2026.
- Reviews originate from Google, TripAdvisor and Facebook. 
- Star ratings range from 1 to 5. 
- No missing or whitespace-only fields were identified in the raw file.
- Site names contain inconsistent capitalisation and surrounding spaces.
- There are 16 additional exact duplicate rows before standardisation.
- No review ID, reviewer ID or precise timestamp is available, limiting the ability to confirm whether matching rows are duplicate submissions. 

## Data dictionary 

| Column | Meaning | Expected type | Validation |
|---|---|---|---|
| site | Site associated with the review | String | Consistent with trading-site names |
| date | Recorded review date; visit date is unconfirmed | Date | Valid, parseable date |
| source | Platform the review came from | String | Inspect and standardise platform labels if needed |
| star_rating | Guest's recorded rating | Integer | Whole number from 1 to 5 |
| review_text | Original written review | String | Preserve wording; check for missing or blank text |

Multiple reviews can share the same site and date.
Therefore, site and date do not uniquely identify a review.

# Standardising site and source names

Site names are standardised to match the site-trading dataset.
Review-source labels are standardised for consistent grouping. 

The original dataset remains unchanged in `df_raw`.
Review text is not modified. 

In [ ]:
df["site"] = (
    df["site"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["source"] = (
    df["source"]
    .astype("string")
    .str.strip()
    .str.lower()
)

display(df["site"].value_counts(dropna=False))
display(df["source"].value_counts(dropna=False))

Provides 5 different sites and 3 different sources - cleaned from raw. 

In [ ]:
expected_sites = {
    "the wellington",
    "the hercules",
    "the sanctuary house",
    "the mad-hatter",
    "the butcher's hook and cleaver"
}

df["site_unrecognised"] = ~df["site"].isin(expected_sites)

display(
    df.loc[df["site_unrecognised"], ["site", "date"]]
)

No unrecognised site names were found after standardisation. This check validates site labels only.

# Date validation

Dates are parsed into a separate datetime column. 
The original date values are retained. 

The recorded date is treated as a review date. It is not assumed to identify when the guest visited the site. 

In [ ]:
df["date_parsed"] = pd.to_datetime(
    df["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

df["date_missing"] = (
    df["date"].isna()
    | df["date"].astype("string").str.strip().eq("").fillna(False)
)

df["date_invalid"] = ( 
    ~df["date_missing"]
    & df["date_parsed"].isna()
)

print("Missing dates:", df["date_missing"].sum())
print("Invalid dates:", df["date_invalid"].sum())

print("Earliest date:", df["date_parsed"].min())
print("Latest date:", df["date_parsed"].max())

display(
    df.loc[
        df["date_missing"] | df["date_invalid"], 
        ["site", "date", "source"]
    ]
)

Errors="coerce" puts unparseable values into NaT in the helper column. The original date column remains available for investigation.

# Star-rating validation

Valid ratings must be whole numbers between 1 and 5. 
Missing, non-numeric, fractional and out-of-range ratings are flagged. 

No ratings are estimated, rounded or replaced. 

In [ ]:
df["rating_numeric"] = pd.to_numeric(
    df["star_rating"],
    errors="coerce"
)

df["rating_missing"] = ( 
    df["star_rating"].isna()
    | df["star_rating"]
    .astype("string")
    .str.strip()
    .eq("")
    .fillna(False)
)

df["rating_valid"] = (
    df["rating_numeric"].notna()
    & df["rating_numeric"].between(1,5)
    & df["rating_numeric"].mod(1).eq(0)
).fillna(False)

df["rating_invalid"] = ( 
    ~df["rating_missing"]
    & ~df["rating_valid"]
)

display(
    df["star_rating"]
    .value_counts(dropna=False)
    .sort_index()
)

display(
    df.loc[
        df["rating_missing"] | df["rating_invalid"],
        ["site", "date", "star_rating"]
    ]
)

# Text completeness 

Site names, review sources and review text are checked for missing values and whitespace-only entries. 

Original review wording, punctuation and capitalisation are preserved. 

In [ ]:
for column in ["site", "source", "review_text"]: 
    df[f"{column}_missing"] = (
        df[column].isna()
        | df[column]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
    )

text_flags = [
    "site_missing", 
    "source_missing",
    "review_text_missing"
]

display(df[text_flags].sum().rename("rows_flagged"))

# Duplicate investigation 

Matching records are identified using site, date, source, star rating and original review text. 

Multiple guests can review the same site on the same date. 
Site and date alone are therefore not a unique review key.

Without a review ID, reviewer ID or precise timestamp, identical records are treated as potential duplicates and retained. 

In [ ]:
review_columns = [
    "site",
    "date",
    "source",
    "star_rating",
    "review_text"
]

# Every row belonging to a matching group 
df["potential_duplicate"] = df.duplicated(
    subset=review_columns,
    keep=False
)

# Additional copies after the first occurrence 
df["duplicate_after_first"] = df.duplicated(
    subset=review_columns,
    keep="first"
)

print(
    "Rows belonging to matching groups:", 
    df["potential_duplicate"].sum()
)

print(
    "Additional matching copies:",
    df["duplicate_after_first"].sum()
)

display(
    df.loc[df["potential_duplicate"], review_columns]
    .sort_values(["site", "date", "source"])
)

# These two flags serve different purposes:
# potential_duplicate: inspect the whole group.
# duplicate_after_first: identify additional copies for a possible sensitivity analysis.

### Cleaning decisions 

- The raw dataset was preserved separately.
- Site and source labels were standardised in the working dataframe. 
- Original review text, date values and star ratings were retained. 
- Parsed dates and numeric ratings were stored in helper columns.
- Missing fields, invalid dates and invalid ratings were flagged. 
- Unrecognised site names were flagged for investigation. 
- Matching review records were flagged as potential duplicates. 
- No rows were removed and no missing values were imputed. 

Potential duplicates remain unresolved because the dataset does not contain identifiers sufficient to establish whether matching reviews represent the same submission. 

# Exploratory chart : rating distribution 

This chart includes potential duplicates.

In [ ]:
plot_df = df.loc[df["rating_valid"]].copy()

rating_counts = (
    plot_df["rating_numeric"]
    .value_counts()
    .reindex([1,2,3,4,5], fill_value=0)
)

fig, ax = plt.subplots(figsize=(8,5)) 

bars = ax.bar(
    rating_counts.index,
    rating_counts.values,
    color="steelblue"
)

ax.bar_label(bars, padding=3)

ax.set_title("Guest Review Rating Distribution")
ax.set_xlabel("Star Rating")
ax.set_ylabel("Number of Records")
ax.set_xticks([1,2,3,4,5])

ax.margins(y=0.12)

plt.tight_layout()
plt.show()

### Interpretation and limitations

This chart shows the distribution of valid recorded star ratings. 
Potential duplicate records remain included. 

The distribution describes submitted reviews, not all guests. 
Guests who leave reviews may differ from those who do not. 

Most common rating in this chart is a 5 star rating with 4 falling behind and 2 & 3 close with each other. 

# Exploratory chart : Review counts by site

This chart shows the number of recorded reviews for each site.

In [ ]:
site_counts = (
    df.loc[~df["site_missing"]]
    .groupby("site")
    .size()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10,5))

bars = ax.barh(
    site_counts.index.to_series().str.split().apply(lambda words: " ".join(w.capitalize() for w in words)),
    site_counts.values,
    color="steelblue"
)

ax.bar_label(bars, padding=3)

ax.set_title("Recorded Reviews by Site")
ax.set_xlabel("Number of Records")
ax.set_ylabel("")

ax.margins(x=0.15)

plt.tight_layout()
plt.show()

## Interpretation and limitations

Review counts show how much recorded feedback is available for each site. 
Potential duplicates remain included. 

A higher review count does not establish higher satisfaction or more guests. 
Rating comparisons should report the number of reviews supporting each result. 

# Final validation and audit summary 

In [ ]:
assert len(df) == len(df_raw), "The row count has changed."

for column in ["date", "star_rating", "review_text"]:
    pd.testing.assert_series_equal(
        df[column],
        df_raw[column]
    )

In [ ]:
audit_summary = pd.DataFrame({
    "metric": [
        "raw_rows",
        "validated_rows",
        "rows_removed",
        "standardised_sites",
        "missing_site_rows",
        "unrecognised_site_rows",
        "missing_source_rows",
        "missing_review_text_rows",
        "missing_date_rows",
        "invalid_date_rows",
        "missing_rating_rows",
        "invalid_rating_rows",
        "rows_in_potential_duplicate_groups",
        "additional_matching_copies"
    ],
    "value": [ 
        len(df_raw),
        len(df),
        len(df_raw) - len(df),
        df["site"].nunique(),
        int(df["site_missing"].sum()),
        int(df["site_unrecognised"].sum()),
        int(df["source_missing"].sum()),
        int(df["review_text_missing"].sum()),
        int(df["date_missing"].sum()),
        int(df["date_invalid"].sum()),
        int(df["rating_missing"].sum()),
        int(df["rating_invalid"].sum()),
        int(df["potential_duplicate"].sum()),
        int(df["duplicate_after_first"].sum())
    ]
})

display(audit_summary)

## Exporting 

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(CLEAN_PATH, index=False)

print(f"Exported {len(df):,} rows to:")
print(CLEAN_PATH) 

# Conclusion and handoff

The guest-review dataset has been prepared for downstream analysis. 

Site and source labels were standardised, dates and ratings were validated and missing fields were checked. The original raw dataset was preserved. Original review text, date values and star ratings were retained, with parsed values stored separately. 

Potential duplicates were flagged but retained because review-level identifiers are unavailable. No rows were removed and no values were imputed. The audit summary records the outstanding quality issues.

The exported dataset will support customer-experience analysis in SQL, reconciliation in Excel and visualisation in Power BI. 

Review dates are not assumed to be visit dates. Relationships between ratings and trading performance will be treated as exploratory associations, not evidence of causation.